# ASVspoof5 Spectrogram Viewer
1. Run the **setup** cell to build the tar index (once per session).
2. Use **`show(utt_id)`** to display the spectrogram and play the audio for any utterance.
3. Use **`show_random(n)`** to pick `n` random utterances.

In [ ]:
import io
import tarfile
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
from IPython.display import Audio, display

# ── Global config ─────────────────────────────────────────────────
REPO_ROOT = Path.cwd()

# Auto-detect data path (server vs local naming)
_CANDIDATES = [
    REPO_ROOT / "data" / "datasets" / "ASVspoof5_tars",   # server
    REPO_ROOT / "data" / "ASV5spoof5_tars",                # local
]
DATA_ROOT = next((p for p in _CANDIDATES if p.is_dir()), _CANDIDATES[0])
PROTOCOLS_DIR = DATA_ROOT / "ASVspoof5_protocols"

SPLIT = "train"           # "train" or "dev"
LABEL_FILTER = None        # None = any, or "spoof" / "bonafide"
SR = 16000

# ReDimNet config
MODEL_REPO = "IDRnD/ReDimNet"
MODEL_NAME = "b6"
MODEL_TRAIN_TYPE = "ptn"
MODEL_DATASET = "vox2"
TARGET_LAYER = "stage4"

MANIFEST_COLS = [
    "speaker_id", "utt_id", "gender", "x1", "x2", "x3",
    "codec", "system_id", "label", "x4",
]

# ── Resolve split-specific paths ──────────────────────────────────
if SPLIT == "train":
    MANIFEST_FILE = PROTOCOLS_DIR / "ASVspoof5.train.tsv"
    TAR_DIR = DATA_ROOT / "ASVspoof5_audio_train_tars"
    TAR_GLOB = "flac_T_*.tar"
elif SPLIT == "dev":
    MANIFEST_FILE = PROTOCOLS_DIR / "ASVspoof5.dev.track_1.tsv"
    TAR_DIR = DATA_ROOT / "ASVspoof5_audio_dev_tars"
    TAR_GLOB = "flac_D_*.tar"
else:
    raise ValueError(f"SPLIT must be 'train' or 'dev', got '{SPLIT}'")

print(f"DATA_ROOT: {DATA_ROOT}")

# ── Load manifest ─────────────────────────────────────────────────
df = pd.read_csv(MANIFEST_FILE, sep=r"\s+", header=None, names=MANIFEST_COLS)
print(f"Manifest: {len(df)} utterances  |  labels: {df['label'].value_counts().to_dict()}")

if LABEL_FILTER:
    df = df[df["label"] == LABEL_FILTER].reset_index(drop=True)
    print(f"After filter ('{LABEL_FILTER}'): {len(df)} utterances")

# ── Build tar index ───────────────────────────────────────────────
print("Building tar index...")
tar_index: dict[str, tuple[Path, tarfile.TarInfo]] = {}

for tar_path in sorted(TAR_DIR.glob(TAR_GLOB)):
    with tarfile.open(tar_path, "r", ignore_zeros=True) as tf:
        while True:
            try:
                member = tf.next()
                if member is None:
                    break
                if member.isfile():
                    tar_index[Path(member.name).stem] = (tar_path, member)
            except tarfile.ReadError:
                break

df = df[df["utt_id"].isin(tar_index)].reset_index(drop=True)
print(f"Indexed {len(tar_index)} files  |  Available in manifest: {len(df)} utterances")

# ── Load ReDimNet model ───────────────────────────────────────────
print("Loading ReDimNet b6...")
redimnet = torch.hub.load(MODEL_REPO, "ReDimNet", model_name=MODEL_NAME, train_type=MODEL_TRAIN_TYPE, dataset=MODEL_DATASET)
redimnet.eval()
print("Model loaded ✓")

In [ ]:
STAGE_NAMES = ["stem", "stage0", "stage1", "stage2", "stage3", "stage4", "stage5"]


def _extract_audio(utt_id: str):
    """Extract raw audio bytes from tar for a given utt_id."""
    tar_path, member_info = tar_index[utt_id]
    with tarfile.open(tar_path, "r", ignore_zeros=True) as tf:
        return tf.extractfile(member_info).read()


def _get_waveform_tensor(raw_bytes):
    """Load raw audio bytes into a torch tensor at 16kHz."""
    waveform, sample_rate = torchaudio.load(io.BytesIO(raw_bytes))
    if sample_rate != SR:
        waveform = torchaudio.functional.resample(waveform, sample_rate, SR)
    return waveform[:1, :].float()


def _run_model(waveform_tensor):
    """Run full model forward, capturing mel + all stage activations."""
    captured = {}
    handles = []

    def make_hook(name):
        def hook_fn(module, inp, out):
            captured[name] = out.detach()
        return hook_fn

    handles.append(redimnet.spec.register_forward_hook(make_hook("mel")))
    for name in STAGE_NAMES:
        stage = getattr(redimnet.backbone, name, None)
        if stage is not None:
            handles.append(stage.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        redimnet(waveform_tensor)

    for h in handles:
        h.remove()
    return captured


def _make_title(utt_id, row):
    title = f"{utt_id}  |  speaker={row['speaker_id']}  |  label={row['label']}"
    if row["label"] == "spoof":
        title += f"  |  system={row['system_id']}"
    return title


def show(*utt_ids: str):
    """Show librosa mel, ReDimNet mel, and all layer activations + audio player."""
    for utt_id in utt_ids:
        if utt_id not in tar_index:
            print(f"'{utt_id}' not found in tar index")
            continue
        rows = df[df["utt_id"] == utt_id]
        if rows.empty:
            print(f"'{utt_id}' not found in manifest")
            continue
        row = rows.iloc[0]
        raw = _extract_audio(utt_id)
        title = _make_title(utt_id, row)

        waveform_np, sr = librosa.load(io.BytesIO(raw), sr=SR, mono=True)
        waveform_t = _get_waveform_tensor(raw)
        captured = _run_model(waveform_t)

        # Row 1: librosa mel + ReDimNet mel
        fig, axes = plt.subplots(1, 2, figsize=(18, 3.5))

        mel = librosa.feature.melspectrogram(
            y=waveform_np, sr=sr, n_mels=64, n_fft=512, hop_length=256, power=2.0,
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        librosa.display.specshow(mel_db, sr=sr, hop_length=256, x_axis="time", y_axis="mel", ax=axes[0])
        axes[0].set_title("Librosa Mel-Spectrogram")
        axes[0].set_xlabel("Time (s)")
        axes[0].set_ylabel("Frequency (Hz)")

        model_mel_np = captured["mel"].squeeze().numpy()
        axes[1].imshow(model_mel_np, aspect="auto", origin="lower", cmap="magma")
        axes[1].set_title(f"ReDimNet Mel ({model_mel_np.shape[0]} x {model_mel_np.shape[1]})")
        axes[1].set_xlabel("Frame")
        axes[1].set_ylabel("Mel bin")

        fig.suptitle(title, fontsize=12)
        plt.tight_layout()
        plt.show()

        # Row 2: all stage activations
        available_stages = [s for s in STAGE_NAMES if s in captured]
        n_stages = len(available_stages)
        fig, axes = plt.subplots(1, n_stages, figsize=(3.5 * n_stages, 4))
        if n_stages == 1:
            axes = [axes]

        for ax, stage_name in zip(axes, available_stages):
            act = captured[stage_name].squeeze().numpy()
            if act.ndim == 1:
                act = act[np.newaxis, :]
            ax.imshow(act, aspect="auto", origin="lower", cmap="viridis")
            ax.set_title(f"{stage_name}\n({act.shape[0]}ch x {act.shape[1]}f)")
            ax.set_xlabel("Frame")
            ax.set_ylabel("Channel")

        fig.suptitle(f"All Layer Activations — {utt_id}", fontsize=12)
        plt.tight_layout()
        plt.show()

        display(Audio(waveform_np, rate=sr))


def show_random(n: int = 5):
    """Show spectrograms + audio players for n random utterances (at least half bonafide)."""
    n_bonafide = (n + 1) // 2
    n_spoof = n - n_bonafide
    df_bon = df[df["label"] == "bonafide"]
    df_spf = df[df["label"] == "spoof"]
    picked = pd.concat([
        df_bon.sample(n=min(n_bonafide, len(df_bon))),
        df_spf.sample(n=min(n_spoof, len(df_spf))),
    ]).sample(frac=1)  # shuffle
    show(*picked["utt_id"].tolist())

In [ ]:
# Show by specific ID
show("T_0000000000")

In [ ]:
# Show 5 random utterances
show_random(5)